# Cache Me Offside: team workbench
**More than a Hack 2027, Phase 1 (ball possession)**

This notebook gets anyone on the team from zero to a validated submission in one **Runtime → Run all**: it pulls our code from GitHub, pulls the match data from our Drive folder, runs our pipeline, checks the file the same way the official scorer does, and saves a copy to Drive with a log line.

**Where things live**

| What | Where | Why |
| --- | --- | --- |
| Code | our GitHub fork, in `cmo/` | the submission needs a public repo that regenerates the file |
| Match data | our Drive folder, `CacheMeOffside/data/` | gated data, never on GitHub |
| Submissions and run log | our Drive folder, `CacheMeOffside/submissions/` and `runs_log.csv` | shared history for the team |

**Deadline:** Tuesday 17 November, 11:59 PM CET (10:59 PM Dublin). Submission form arrives by email by 14 November.

## Rules for everyone using this notebook
1. **No match data goes into any AI tool**, including Colab's built-in Gemini. Before you start, go to **Tools → Settings → AI assistance** and turn the generative AI features off. Don't click "Explain error" on a failed cell. The competition terms forbid sending the data, or any extract of it, to a hosted LLM.
2. **Share the Drive folder only with teammates who have registered** under our team name and accepted the data terms.
3. **Clear outputs before saving this notebook to GitHub** (Edit → Clear all outputs). Our repo is public.
4. **The public 5 minutes are a pipeline check, not a target.** Never pick settings because they score better there. It's against the rules and it picks the wrong settings anyway.
5. **Hand-labelling this match is banned**, in any amount, including to train or validate a model.

**First-time setup**
- *Owner, once:* in My Drive, create a folder `CacheMeOffside`, and inside it a folder `data`. Unzip the data bundle into `data` keeping its names, so you get `data/tactical/BAR-ATH_tactical_atd.txt` and friends. Share `CacheMeOffside` with registered teammates.
- *Teammates, once:* open **Shared with me**, right-click `CacheMeOffside` → **Organise → Add shortcut → My Drive**.
- *Everyone:* fill in **Settings** below, then **Runtime → Run all**. Re-run whenever you push new code.

In [ ]:
# @title 1. Settings { display-mode: "form" }
# @markdown Our fork on GitHub (the public submission repo):
GITHUB_REPO = "https://github.com/asynctdm/more-than-a-hack-2027.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
# @markdown Folder in My Drive holding `data/` (a shortcut is fine):
DRIVE_FOLDER = "CacheMeOffside"  # @param {type:"string"}
# @markdown Submission file name, no spaces (the scoreboard row is named after it):
TEAM_FILE_NAME = "CacheMeOffside"  # @param {type:"string"}
# @markdown Who is running this (goes in the run log):
YOUR_NAME = ""  # @param {type:"string"}
# @markdown Options:
REBUILD_BASELINES = False  # @param {type:"boolean"}
INSTALL_KLOPPY = False  # @param {type:"boolean"}
COPY_VIDEO = False  # @param {type:"boolean"}

import csv, datetime, hashlib, os, shutil, subprocess, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
REPO = BASE / "repo"
DRIVE_ROOT = Path("/content/drive/MyDrive") if IN_COLAB else BASE / "drive"
TEAM_DIR = DRIVE_ROOT / DRIVE_FOLDER
DATA_SRC = TEAM_DIR / "data"
SUBMISSIONS = TEAM_DIR / "submissions"
LOCAL_DATA = REPO / "data"
OUT_CSV = REPO / "evaluation" / "results" / f"{TEAM_FILE_NAME}.csv"
PY = sys.executable

assert "asynctdm" not in GITHUB_REPO, "Set GITHUB_REPO to the team's fork first."
assert TEAM_FILE_NAME and " " not in TEAM_FILE_NAME, "TEAM_FILE_NAME needs a value with no spaces."

os.environ["PYTHONUNBUFFERED"] = "1"        # so long scripts print as they go
os.environ["MTAH_DATA_DIR"] = str(LOCAL_DATA)


def sh(cmd, cwd=None, check=True, echo=True, show=True):
    """Run a shell command, print its output live, stop here if it fails."""
    if echo:
        print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=str(cwd or REPO), text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    for line in proc.stdout:
        if show:
            print(line, end="", flush=True)
        lines.append(line)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return "".join(lines)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


print("Running in", "Colab" if IN_COLAB else "a local Jupyter", "| repo will be at", REPO)

In [ ]:
# @title 2. Get the code
if (REPO / ".git").exists():
    sh(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
else:
    sh(f"git clone --branch {BRANCH} {GITHUB_REPO} {REPO}", cwd=BASE)

COMMIT = sh("git rev-parse --short HEAD", echo=False, show=False).strip()
for path in (REPO, REPO / "starter", REPO / "evaluation"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("Code is at commit", COMMIT)

In [ ]:
# @title 3. Install
# Colab already has numpy and pandas, which is all the scorer and the baseline need.
# Our pipeline's extra packages go, pinned, in cmo/requirements.txt.
# (Don't install requirements-starter.txt here: it pulls in Jupyter, which can break Colab.)
if INSTALL_KLOPPY:
    sh(f"{PY} -m pip install -q kloppy==3.18.0 matplotlib")
if (REPO / "cmo" / "requirements.txt").exists():
    sh(f"{PY} -m pip install -q -r cmo/requirements.txt")

import numpy as np
import pandas as pd
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# @title 4. Connect the data
# Copies the data from Drive onto this machine's disk, because reading 199,319 lines
# straight from Drive is slow. Videos are skipped unless COPY_VIDEO is ticked.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

HAVE_DATA = False
if not TEAM_DIR.exists():
    print(f"No folder '{DRIVE_FOLDER}' in My Drive.")
    print("Teammates: Shared with me -> right-click the folder -> Organise -> Add shortcut -> My Drive.")
else:
    SUBMISSIONS.mkdir(exist_ok=True)
    n_copied = 0
    for src in sorted(DATA_SRC.rglob("*")) if DATA_SRC.exists() else []:
        if not src.is_file() or (src.suffix == ".mp4" and not COPY_VIDEO):
            continue
        dst = LOCAL_DATA / src.relative_to(DATA_SRC)
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            n_copied += 1
    print(f"Copied {n_copied} new or changed file(s) from Drive.")

    from games import asset_paths
    paths = asset_paths(data_dir=str(LOCAL_DATA))
    missing = [name for name in ("atd_meta", "atd_raw", "smart_tagging")
               if not os.path.isfile(paths[name])]
    HAVE_DATA = not missing
    if missing:
        print("Missing:", ", ".join(Path(paths[name]).name for name in missing))
        print(f"Expected them under {DATA_SRC}/tactical/ in Drive. If the zip made an extra")
        print("top-level folder, move its contents up so the path matches.")

print("Data ready." if HAVE_DATA else "No data yet: the scorer still runs, the pipeline cells will skip.")

In [ ]:
# @title 5. Sanity-check the data
if HAVE_DATA:
    import load_atd
    import load_smart_tagging

    alignment = load_atd.check_alignment_raw()
    print("Tracking:", alignment["n_records"], "records, frame_id",
          "{}..{}".format(*alignment["frame_id_range"]),
          "->", "OK" if alignment["agrees"] else "MISMATCH, tell the organisers")

    time_base = load_smart_tagging.check_time_base(load_smart_tagging.load())
    print("Smart Tagging clock:", "OK" if time_base["agrees"] else "MISMATCH, tell the organisers")
else:
    print("Skipped: no data.")

In [ ]:
# @title 6. Scorer check (the official script, public 5 minutes)
if REBUILD_BASELINES and HAVE_DATA:
    sh(f"{PY} starter/baseline.py")    # a minute or two
sh(f"{PY} evaluation/run_evaluation.py --no-detail")

## Our pipeline
The contract every version of our code keeps:

- `python cmo/run.py --out <path>` writes the full 199,319-row `frame,label` CSV to `<path>`.
- It reads the match data only from the data folder (`games.asset_paths()` finds it).
- Same data in, same file out, every time. Fix any random seeds.
- Its extra packages, pinned, go in `cmo/requirements.txt`.

In [ ]:
# @title 7. Run our pipeline
HAVE_SUBMISSION = False
if not (REPO / "cmo" / "run.py").exists():
    print("No cmo/run.py in the repo yet. Create it, push it, then re-run from step 2.")
elif not HAVE_DATA:
    print("Skipped: no data.")
else:
    started = time.time()
    sh(f"{PY} cmo/run.py --out {OUT_CSV}")
    HAVE_SUBMISSION = OUT_CSV.exists()
    print(f"Took {time.time() - started:.0f} s.", "Wrote" if HAVE_SUBMISSION else "Did NOT write", OUT_CSV.name)

In [ ]:
# @title 8. Validate and score it
if HAVE_SUBMISSION:
    sh(f"{PY} evaluation/validate_submission.py {OUT_CSV}")
    sh(f"{PY} evaluation/run_evaluation.py --no-detail")
    print("\nReminder: this score is a pipeline check on 17 possessions, not a target.")
else:
    print("Skipped: no submission file.")

In [ ]:
# @title 9. Reproducibility check (rule 3: the repo must regenerate the same file)
REPRODUCIBLE = None
if HAVE_SUBMISSION:
    rerun = BASE / "rerun" / OUT_CSV.name      # kept out of evaluation/results so it isn't scored
    rerun.parent.mkdir(exist_ok=True)
    sh(f"{PY} cmo/run.py --out {rerun}")
    REPRODUCIBLE = sha256(OUT_CSV) == sha256(rerun)
    if REPRODUCIBLE:
        print("Identical on a second run.")
    else:
        first, second = pd.read_csv(OUT_CSV), pd.read_csv(rerun)
        changed = int((first["label"] != second["label"]).sum())
        print(f"NOT identical: {changed} rows differ. Look for unseeded randomness or set ordering.")
else:
    print("Skipped: no submission file.")

In [ ]:
# @title 10. Save to Drive and log the run
if HAVE_SUBMISSION and TEAM_DIR.exists():
    import possession_metrics as pm
    from games import GAME, half_video_blocks

    codes = pm.label_codes(pm.read_label_csv(OUT_CSV))
    graded = [(first, last + 1) for _, first, last in half_video_blocks(GAME)]
    filled = pm.fill_abstentions(codes, graded)          # same X handling as the scorer
    spells = pm.possession_spells(filled, graded)
    in_graded = np.concatenate([codes[a:b] for a, b in graded])
    alive = int(np.isin(np.concatenate([filled[a:b] for a, b in graded]), (pm.H, pm.A)).sum())
    mean_s = alive / 25.0 / len(spells) if spells else 0.0
    share = {name: 100.0 * np.mean(in_graded == idx) for idx, name in enumerate(pm.LABELS)}

    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%SZ")
    saved = SUBMISSIONS / f"{TEAM_FILE_NAME}_{stamp}_{COMMIT}.csv"
    shutil.copy2(OUT_CSV, saved)

    log = TEAM_DIR / "runs_log.csv"
    row = {"utc": stamp, "who": YOUR_NAME or "?", "commit": COMMIT,
           "file_sha256": sha256(OUT_CSV)[:12], "possessions": len(spells),
           "mean_possession_s": round(mean_s, 1),
           **{f"pct_{k}": round(v, 1) for k, v in share.items()},
           "reproducible": REPRODUCIBLE, "saved_as": saved.name}
    new_log = not log.exists()
    with open(log, "a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row))
        if new_log:
            writer.writeheader()
        writer.writerow(row)

    print(f"Saved {saved.name} to Drive.")
    print(f"Graded halves: {len(spells)} possessions, mean {mean_s:.1f} s,",
          ", ".join(f"{k} {v:.1f}%" for k, v in share.items()))
    print(pd.read_csv(log).tail(5).to_string(index=False))
else:
    print("Skipped: no submission file or no Drive folder.")

## Optional: push code changes from Colab
Only if you edited code here instead of on your laptop. One-time setup:
1. GitHub → Settings → Developer settings → **Fine-grained token**, access to **our fork only**, permission **Contents: read and write**.
2. In Colab, click the **key icon** in the left sidebar, add a secret named `GITHUB_TOKEN` with the token, and switch on notebook access. Never paste the token into a cell.
3. The fork owner adds you as a collaborator (repo → Settings → Collaborators).

The cell commits only the paths you list, and refuses to push anything under `data/`.

In [ ]:
# @title 11. Push code changes (optional)
PUSH = False  # @param {type:"boolean"}
GITHUB_USERNAME = ""  # @param {type:"string"}
GIT_EMAIL = ""  # @param {type:"string"}
COMMIT_MESSAGE = ""  # @param {type:"string"}
PATHS_TO_COMMIT = "cmo NOTES.md"  # @param {type:"string"}

if PUSH:
    assert IN_COLAB, "Push from your laptop with git when running locally."
    assert GITHUB_USERNAME and GIT_EMAIL and COMMIT_MESSAGE, "Fill in username, email and message."
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")

    existing = " ".join(p for p in PATHS_TO_COMMIT.split() if (REPO / p).exists())
    sh(f"git add -- {existing}")
    staged = sh("git diff --cached --name-only", echo=False, show=False).split()
    blocked = [p for p in staged if p.startswith("data/") or p.endswith((".mp4", ".pkl", ".npz"))]
    if blocked:
        sh("git reset -q")
        raise RuntimeError(f"Refusing to push data files: {blocked}")
    if not staged:
        print("Nothing to commit.")
    else:
        sh(f'git -c user.name="{GITHUB_USERNAME}" -c user.email="{GIT_EMAIL}" commit -m "{COMMIT_MESSAGE}"')
        remote = GITHUB_REPO.replace("https://", f"https://{GITHUB_USERNAME}:{token}@")
        result = subprocess.run(["git", "push", remote, f"HEAD:{BRANCH}"], cwd=REPO,
                                capture_output=True, text=True)
        print((result.stdout + result.stderr).replace(token, "***"))
        print("Pushed." if result.returncode == 0 else "Push failed, see above.")
else:
    print("Push is off.")